In [7]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

MODEL_PATH = "zai-org/GLM-OCR"
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "You are a helpful assistant for recognizing text in images. Extract all text faithfully."
            },
            {
                "type": "image",
                 "url": "http://localhost:8080/ready-for-ocr/2026-03-21T1428_NB_generated-1-left-orig-aspect.png"
            },
            {
                "type": "text",
                "text": "Text Recognition:"
            }
        ],
    }
]
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
)
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)
inputs.pop("token_type_ids", None)
generated_ids = model.generate(**inputs, max_new_tokens=8192, repetition_penalty=1.3)
output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
print(output_text)

The image processor of type `Glm46VImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

Aarskiftet.

Me ynskjer å'le være tingarar, tillitmenn og meldarar eit god og signerikt nyår.

Det er vanleg skikk ved nyärsskiften Å gjera opp reknska-pet för äret som gjelkk. Og deretter å koma med nokre meir eller minder vel grunngjevne funderingar og spådomar for det nye året.
Men det er ikkje so lett å gjERA opp et rett reknkap for ÄRET 1947, og enno verre dette bil à setja opp noko budsjttt for 1948.
Ingen veit kva ÆRET førr med seg. Me får berre vona det beste, og ikkje tapa trui på ei jos-sare og fredeleg framtid.
Megyttar dette hove ved ärrs-kifte å seia fra til alle som spør om ikkje Sogns Avis snart kjem utatt 2 gonger om vika: Jau,det er sjølvsagt meiningsi.Men ttingarane lyt tru oss när ne seier at hittil har det vor uräd.Grunnnane til det er fleire,og me skal ikkje koma nermare inn på dei her.Мe liver likevel i von om at det ska retta seg i året som kjem.Det kan ikkje seiast å vera noko troyst,mene me nemner det likveel; det er fleire blad av dei som varrt stoggä i krigså

In [ ]:
# --- Fine-tuned model (5,000 samples, 3 epochs) ---
# Compare output against the base model cell above on the same image.

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# Free the base model from GPU memory before loading the fine-tuned one
try:
    del model, processor, inputs, generated_ids
    torch.cuda.empty_cache()
    print("Cleared base model from GPU memory.")
except NameError:
    pass

FINETUNED_PATH = "/mnt/data/development/GLM-OCR-Norwegian-Nynorsk/LLaMA-Factory/saves/glm-ocr-nynorsk/full/sft"

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "You are a helpful assistant for recognizing text in images. Extract all text faithfully."
            },
            {
                "type": "image",
                "url": "http://localhost:8080/ready-for-ocr/2026-03-21T1428_NB_generated-1.png"
            },
            {
                "type": "text",
                "text": "Text Recognition:"
            }
        ],
    }
]

processor = AutoProcessor.from_pretrained(FINETUNED_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    FINETUNED_PATH,
    torch_dtype="auto",
    device_map="auto",
)

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)
inputs.pop("token_type_ids", None)

generated_ids = model.generate(**inputs, max_new_tokens=8192)
output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
print(output_text)

Cleared base model from GPU memory.


Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

Aarskiftet.

Me ynskjer å'le våre tingarar, tillitsmenn og meldarar eit godt og signerikt nyår.

Det er vanleg skikk ved nyårsskiftet å gjera opp reknskapet for året som gjekk. og deretter å koma med nokre meir eller minder vel grunngjevne funderingar og spådomar for det nye året.

Men det er ikkje so lett å gjera opp eit rett reknskap for året 1947, og enno verre dette bil å setja opp noko budsjitt for 1948.

Ingen veit kva året fører med seg. Me får berre vona det beste, og ikkje tapa tru i ei ljosare og fredeleg framtid.

Mør nyttar dette hove ved årsskifte å seia frå til åle som spør om ikkje Sogns Avis snart kjem utatt 2 gonger om vika: Jau, det er sjølvsagt meiningi. Men tingarane lyt tru oss når me seier at hittil har det vore uråd. Grunnane til det er fleire, og me skal ikkje koma nær møre inn på dei her. Me liver likevel i von om at det skal retta seg i året som kjem. Det kan ikkje seiast å vera noko tròyst, men me nemner det likevel: det er fleire blad av dei som vart stogga 

In [4]:
# --- Fine-tuned model (5,000 samples, 3 epochs) ---
# Quadrant Splitting & Inference Pipeline

import torch
import requests
from io import BytesIO
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

# Free the base model from GPU memory before loading the fine-tuned one
try:
    del model, processor, inputs, generated_ids
    torch.cuda.empty_cache()
    print("Cleared base model from GPU memory.")
except NameError:
    pass

FINETUNED_PATH = "/mnt/data/development/GLM-OCR-Norwegian-Nynorsk/LLaMA-Factory/saves/glm-ocr-nynorsk/full/sft"
IMAGE_URL = "http://localhost:8080/ready-for-ocr/2026-03-21T1428_NB_generated-p1-1.jpeg"

def get_overlapping_quadrants(image_url: str, overlap_pct: float = 0.15) -> dict:
    """Fetches an image and slices it into 4 overlapping quadrants."""
    print(f"Fetching image from: {image_url}")
    response = requests.get(image_url)
    response.raise_for_status()
    img = Image.open(BytesIO(response.content)).convert("RGB")
    
    w, h = img.size
    overlap_x = int((w / 2) * overlap_pct)
    overlap_y = int((h / 2) * overlap_pct)
    
    mid_x, mid_y = w // 2, h // 2

    print(f"Original size: {w}x{h}. Slicing with {overlap_pct*100}% overlap...")
    
    quadrants = {
        "Q1_Top_Left": img.crop((0, 0, mid_x + overlap_x, mid_y + overlap_y)),
        "Q2_Top_Right": img.crop((mid_x - overlap_x, 0, w, mid_y + overlap_y)),
        "Q3_Bottom_Left": img.crop((0, mid_y - overlap_y, mid_x + overlap_x, h)),
        "Q4_Bottom_Right": img.crop((mid_x - overlap_x, mid_y - overlap_y, w, h))
    }
    
    return quadrants

print("Loading processor and model...")
processor = AutoProcessor.from_pretrained(FINETUNED_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    FINETUNED_PATH,
    torch_dtype="auto",
    device_map="auto",
)

# 1. Get the 4 image quadrants
quadrants = get_overlapping_quadrants(IMAGE_URL)
ocr_results = {}

# 2. Run inference on each quadrant
for quad_name, pil_img in quadrants.items():
    print(f"\nProcessing {quad_name} ({pil_img.size[0]}x{pil_img.size[1]} px)...")
    
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "You are a helpful assistant for recognizing text in images. Extract all text faithfully."
                },
                {
                    "type": "image",
                    "image": pil_img  # Passing the PIL image directly
                },
                {
                    "type": "text",
                    "text": "Text Recognition:"
                }
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    inputs.pop("token_type_ids", None)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=4096)
        
    output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    ocr_results[quad_name] = output_text.strip()
    print(f"Finished {quad_name}.")

# 3. Assemble the prompt for the LLM stitcher
stitching_prompt = f"""You are an expert document reconstruction engine. I am providing you with four text transcriptions representing four overlapping quadrants of a single page from a 1939 Norwegian newspaper. 

Task:
1. Reconstruct the full text of the page.
2. Maintain the correct logical reading order, flowing down the columns natively.
3. Use the overlapping text at the boundaries of the quadrants to perfectly stitch the articles together. Eliminate the duplicated overlap text in your final output.
4. Do not summarize, translate, or correct the historical spelling. Output ONLY the unified, merged Norwegian text.

Inputs:

<Q1_Top_Left>
{ocr_results["Q1_Top_Left"]}
</Q1_Top_Left>

<Q2_Top_Right>
{ocr_results["Q2_Top_Right"]}
</Q2_Top_Right>

<Q3_Bottom_Left>
{ocr_results["Q3_Bottom_Left"]}
</Q3_Bottom_Left>

<Q4_Bottom_Right>
{ocr_results["Q4_Bottom_Right"]}
</Q4_Bottom_Right>
"""

print("\n" + "="*50)
print("READY FOR LLM STITCHING:")
print("="*50)
print(stitching_prompt)

# Optional: Save the prompt to a text file
with open("stitching_prompt_output.txt", "w", encoding="utf-8") as f:
    f.write(stitching_prompt)
print("\nSaved prompt to stitching_prompt_output.txt")

Loading processor and model...


Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

Fetching image from: http://localhost:8080/ready-for-ocr/2026-03-21T1428_NB_generated-p1-1.jpeg
Original size: 2862x4143. Slicing with 15.0% overlap...

Processing Q1_Top_Left (1645x2381 px)...
Finished Q1_Top_Left.

Processing Q2_Top_Right (1645x2381 px)...
Finished Q2_Top_Right.

Processing Q3_Bottom_Left (1645x2382 px)...
Finished Q3_Bottom_Left.

Processing Q4_Bottom_Right (1645x2382 px)...
Finished Q4_Bottom_Right.

READY FOR LLM STITCHING:
You are an expert document reconstruction engine. I am providing you with four text transcriptions representing four overlapping quadrants of a single page from a 1939 Norwegian newspaper. 

Task:
1. Reconstruct the full text of the page.
2. Maintain the correct logical reading order, flowing down the columns natively.
3. Use the overlapping text at the boundaries of the quadrants to perfectly stitch the articles together. Eliminate the duplicated overlap text in your final output.
4. Do not summarize, translate, or correct the historical spell

In [ ]:
# --- Quadrant OCR → Stitching Prompt → JSON (via external LLM) ---
# Splits the page into 4 overlapping quadrants, runs plain-text OCR on each,
# then builds a stitching+JSON prompt to be sent to a capable LLM separately.

import json
import torch
import requests
from io import BytesIO
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

# Free any existing model from GPU memory
try:
    del model, processor, inputs, generated_ids
    torch.cuda.empty_cache()
    print("Cleared previous model from GPU memory.")
except NameError:
    pass

FINETUNED_PATH = "/mnt/data/development/GLM-OCR-Norwegian-Nynorsk/LLaMA-Factory/saves/glm-ocr-nynorsk/full/sft"
IMAGE_URL = "http://localhost:8080/ready-for-ocr/2026-03-21T1428_NB_generated-p1-1.jpeg"

# JSON schema the stitching LLM should produce for each article
JSON_SCHEMA = '[{"title": "", "author": "", "body": ""}]'

def get_overlapping_quadrants(image_url: str, overlap_pct: float = 0.15) -> dict:
    """Fetches an image and slices it into 4 overlapping quadrants."""
    print(f"Fetching image from: {image_url}")
    response = requests.get(image_url)
    response.raise_for_status()
    img = Image.open(BytesIO(response.content)).convert("RGB")

    w, h = img.size
    overlap_x = int((w / 2) * overlap_pct)
    overlap_y = int((h / 2) * overlap_pct)
    mid_x, mid_y = w // 2, h // 2

    print(f"Original size: {w}x{h}. Slicing with {overlap_pct*100}% overlap...")

    quadrants = {
        "Q1_Top_Left": img.crop((0, 0, mid_x + overlap_x, mid_y + overlap_y)),
        "Q2_Top_Right": img.crop((mid_x - overlap_x, 0, w, mid_y + overlap_y)),
        "Q3_Bottom_Left": img.crop((0, mid_y - overlap_y, mid_x + overlap_x, h)),
        "Q4_Bottom_Right": img.crop((mid_x - overlap_x, mid_y - overlap_y, w, h)),
    }
    return quadrants

print("Loading processor and model...")
processor = AutoProcessor.from_pretrained(FINETUNED_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    FINETUNED_PATH, torch_dtype="auto", device_map="auto",
)

# 1. Get the 4 image quadrants
quadrants = get_overlapping_quadrants(IMAGE_URL)
ocr_results = {}

# 2. Run plain-text OCR on each quadrant (what the model was trained to do)
for quad_name, pil_img in quadrants.items():
    print(f"\nProcessing {quad_name} ({pil_img.size[0]}x{pil_img.size[1]} px)...")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "You are a helpful assistant for recognizing text in images. Extract all text faithfully.",
                },
                {"type": "image", "image": pil_img},
                {"type": "text", "text": "Text Recognition:"},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    inputs.pop("token_type_ids", None)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=4096, repetition_penalty=1.3)

    output_text = processor.decode(
        generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    ocr_results[quad_name] = output_text
    print(f"Finished {quad_name}.")

# 3. Build the stitching + JSON conversion prompt for an external LLM
stitching_prompt = f"""You are an expert document reconstruction engine. I scanned a single newspaper page in four overlapping quadrants and ran OCR on each quadrant. The OCR output is plain text.

Your task:
1. Read the four plain-text transcriptions below.
2. Stitch them into a single coherent page, using the overlapping text at quadrant boundaries to de-duplicate and merge correctly.
3. Identify individual newspaper articles within the merged text.
4. Output the result as a single valid JSON array, where each element represents one article with the schema: {JSON_SCHEMA}
5. Maintain the correct reading order (top-left → top-right → bottom-left → bottom-right, column by column).
6. Do not summarize, translate, or correct the historical Norwegian spelling.
7. Output ONLY the final valid JSON array — no commentary, no markdown fences.

Inputs:

<Q1_Top_Left>
{ocr_results["Q1_Top_Left"]}
</Q1_Top_Left>

<Q2_Top_Right>
{ocr_results["Q2_Top_Right"]}
</Q2_Top_Right>

<Q3_Bottom_Left>
{ocr_results["Q3_Bottom_Left"]}
</Q3_Bottom_Left>

<Q4_Bottom_Right>
{ocr_results["Q4_Bottom_Right"]}
</Q4_Bottom_Right>
"""

print("\n" + "=" * 50)
print("STITCHING PROMPT READY — send to a capable LLM (e.g. Claude, GPT-4)")
print("=" * 50)

with open("stitching_prompt_json_output.txt", "w", encoding="utf-8") as f:
    f.write(stitching_prompt)
print(f"Saved prompt to stitching_prompt_json_output.txt ({len(stitching_prompt)} chars)")

# GLM-OCR: Maximising Results — Reference & Best Practices

**Source:** [2026 Complete Guide: How to Use GLM-OCR for Next-Gen Document Understanding](https://dev.to/sienna/the-complete-2026-guide-building-interactive-dashboards-with-a2ui-rizzcharts-538j)  
*(Published on dev.to by Sienna, Feb 2026)*

---

## What the article tells us about GLM-OCR

GLM-OCR is a **0.9B-parameter multimodal model** built on the GLM-V / CogViT architecture. Key facts relevant to this project:

| Property | Value |
|---|---|
| OmniDocBench V1.5 score | **94.62** (SOTA in its class) |
| Throughput | ~1.86 PDF pages/second |
| Output formats | Markdown, JSON, LaTeX |
| Languages | 100+, including Norwegian |
| License | Apache-2.0 (open weights) |

The model uses **Multi-Token Prediction (MTP)**: rather than rigidly copying visual glyphs, it uses surrounding context to infer and correct tokens during generation. This is particularly valuable for the degraded historical newspaper scans we are working with.

---

## `PRECISION_MODE_ON` — what is it?

The article notes:

> *"The official site highlights a `PRECISION_MODE_ON`, claiming up to 99.9% precision in that mode."*

⚠️ **Important caveat:** `PRECISION_MODE_ON` is a feature of the **hosted API at glmocr.com**, not an argument in the HuggingFace `transformers` interface. The article cites the official site ([glmocr.com](https://glmocr.com)) — exact metric definitions are not publicly documented, so treat "99.9%" as a marketing claim rather than a reproducible benchmark.

The article is clear on when to use it:

> *"Turn on `PRECISION_MODE_ON` for: legal contracts, financial statements, regulatory filings — accept the extra latency in exchange for maximum accuracy."*

For **local inference** (what we do here), the equivalent levers are listed in the next section.

## Maximising results with local inference

### 1. Image preprocessing (biggest lever)

The article is explicit:

> *"For low-DPI or heavily skewed scans, apply: binarization, de-skewing, noise reduction — this helps the visual encoder and improves downstream structure detection."*

For our Norwegian newspaper scans, this means the pipeline in `scans/preprocess.py` is critical **before** feeding pages to GLM-OCR:

- **pdftoppm at 600 DPI** — ensures the CogViT encoder has enough pixels to resolve body text (~8–10 pt)
- **Bilateral filter** — removes paper grain while preserving ink edges (better than Gaussian for OCR)
- **CLAHE** — local contrast enhancement for faded ink, without blowing out the background
- **De-skew** (cell 1 above) — sub-degree correction before any of the above

Do **not** hard-binarize (black/white threshold) for GLM-OCR: the model handles grayscale well and MTP benefits from the grey ink-gradient information. Binarization can destroy stroke-weight cues the CogViT encoder uses.

---

### 2. Generation parameters

| Parameter | Our current value | Effect |
|---|---|---|
| `max_new_tokens` | 8192 | Sets the ceiling; pages average ~6 000 tokens total |
| `repetition_penalty` | 1.3 | Suppresses looping — critical for long newspaper pages |
| `do_sample` | False (default) | Greedy decoding is more accurate for OCR than sampling |

The **`repetition_penalty`** is the local analogue of `PRECISION_MODE_ON`: it forces the model to commit to new tokens rather than drifting into repetitive loops — a common failure mode on dense, visually similar newspaper columns. The finetuned-model output above already shows this working.

If you see looping on a specific scan, try raising `repetition_penalty` to `1.4`–`1.5` before anything else.

---

### 3. Prompt wording

The article does not elaborate on prompting, but from our own experiments the system-prompt should:
- Name the language explicitly: *"All text is in Norwegian nynorsk"*
- Say *"Extract all text faithfully"* rather than *"summarise"* or *"describe"* — the verb matters
- Avoid asking for Markdown formatting: for plain newspaper text, a structured prompt biases the model toward inventing headings

---

### 4. Combine with an LLM for post-processing

The article recommends:

> *"Use GLM-OCR for reliable structure extraction, then feed its output into a general-purpose LLM for: summaries, risk flags, Q&A and report generation."*

For this project the natural next step is a spelling/normalisation pass: GLM-OCR occasionally produces plausible-looking but wrong nynorsk forms (see `<font color="blue">` artefact in the output above). A small nynorsk LLM or rule-based normaliser as a second stage would clean these up without retraining.

---

### 5. What the fine-tune adds

Our fine-tuned model (5 000 samples, 3 epochs) already handles Norwegian-specific challenges the base model does not:
- Correct nynorsk orthography over bokmål guesses
- Newspaper multi-column layout without hallucinating column breaks as paragraph breaks
- Historical spelling variants common in mid-20th century Nynorsk press